# Session 2 — I-VT filter, AOI/TOI metrics, heatmaps & gaze plots  
**Duration:** 1.5 hours

### Learning goals
1. Implement **I-VT** (velocity-threshold identification) on real Tobii samples.
2. Compare I-VT events with Tobii’s `Eye movement type` labels.
3. Assign **AOIs**, compute dwell / TTFF-style metrics, think in **TOIs**.
4. Draw **fixation-count** and **duration** heatmaps and a **Tobii-like gaze plot** on `Stimuli/Decision Making`.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else SESSION_DIR / "workshop"
sys.path.insert(0, str(WORKSHOP_DIR))

from analysis.paths import data_path, stimuli_path
from analysis.ivt import ivt_classify, summarize_ivt_events
from analysis.aoi import load_default_food_aois, hit_test_rectangles, aoi_metrics_from_fixations
from analysis.viz import fixation_heatmap, gaze_plot_tobii_like

## 1. What I-VT is doing (concept, 10 min)

For each sample, estimate speed ≈ distance / Δt.

- If speed **< threshold** → provisional **fixation**
- If speed **≥ threshold** → **saccade**
- Drop fixation runs shorter than a minimum duration

Threshold units matter: we use **pixels/second** on 1366×768 stimuli.


## 2. Prepare gaze samples (15 min)

In [ ]:
raw = pd.read_csv(data_path("food_decision_making", "Food_Decision_Making_Teaching_Sample.csv"))
gaze = raw.query("Sensor == 'Eye Tracker'").copy()
gaze = gaze.dropna(subset=["Gaze point X", "Gaze point Y", "Recording timestamp"])
# Tobii recording timestamps are microseconds in this export
# In THIS export, Recording timestamp is milliseconds (≈17 ms steps ≈ 60 Hz).
# Always print median Δt before choosing an I-VT threshold.
gaze["time_s"] = gaze["Recording timestamp"].astype(float) / 1e3
gaze = gaze.sort_values("time_s")
print("median Δt (s) =", gaze["time_s"].diff().median())
# Focus on one food image TOI/stimulus for a clean demo
stim = "cake"
g = gaze[gaze["Presented Stimulus name"] == stim].copy()
print("samples on", stim, ":", len(g))
g[["time_s", "Gaze point X", "Gaze point Y", "Eye movement type"]].head()

## 3. Run I-VT (20 min)

In [ ]:
classified = ivt_classify(
    g["time_s"], g["Gaze point X"], g["Gaze point Y"],
    velocity_threshold=5000,  # px/s starting point for this screen; try 1000–8000
    min_fixation_duration_s=0.06,
)
classified["label"].value_counts()

In [ ]:
events = summarize_ivt_events(classified)
fix = events.query("label == 'fixation'").copy()
sac = events.query("label == 'saccade'").copy()
print(fix.head())
print({"n_fix": len(fix), "n_sac": len(sac), "mean_fix_s": fix["duration_s"].mean()})

In [ ]:
# Compare with Tobii vendor labels on the same samples
cmp = g[["Eye movement type"]].copy()
cmp["ivt"] = classified["label"].to_numpy()
pd.crosstab(cmp["Eye movement type"], cmp["ivt"])

## 4. AOIs & TOIs (20 min)

- **AOI** = region on the stimulus (food picture, buy, not-buy).
- **TOI** = time window of interest (here: while `cake` is on screen).

Teaching AOIs are approximate rectangles — refine them live if needed.


In [ ]:
aois = load_default_food_aois(stim)
aois

In [ ]:
fix["aoi"] = hit_test_rectangles(fix["centroid_x"], fix["centroid_y"], aois)
fix["start_s"] = fix["start_s"] - fix["start_s"].min()  # TOI-relative optional
aoi_table = aoi_metrics_from_fixations(fix, toi=stim)
aoi_table

## 5. Heatmaps & Tobii-like gaze plot (25 min)

In [ ]:
stim_img = stimuli_path("Decision Making", f"{stim}.png")
assert stim_img.exists(), stim_img

# For heatmap helpers, use columns x/y/duration_s
plot_df = fix.rename(columns={"centroid_x": "x", "centroid_y": "y"})

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
fixation_heatmap(plot_df, stim_img, ax=axes[0], title="Fixation COUNT heatmap")
fixation_heatmap(plot_df, stim_img, weight_col="duration_s", ax=axes[1], title="Fixation DURATION heatmap")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
gaze_plot_tobii_like(plot_df, stim_img, ax=ax, title=f"Gaze plot — {stim}")
plt.show()

## 6. Practice / discussion (10 min)

1. Change `velocity_threshold` to 1000 and 8000. What happens to fixation count?
2. Re-run for `stim = "pizza"`.
3. Debate: when would you prefer **vendor classification** vs **your I-VT**?

### Exit ticket
Paste your AOI metrics table for one stimulus and one sentence interpreting dwell on `buy` vs `food-pic`.
